In [6]:
import pandas as pd
import numpy as np

import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report


In [8]:
df = pd.read_csv("/content/dual_ai_brain_cancer_dataset (1).csv")

print("Dataset shape:", df.shape)
print(df.head())
print(df.columns)

Dataset shape: (617, 20)
                             Patient_ID Age  Gender  Headache_Severity  \
0  0078b0c4-68a9-483b-9aab-61156d263213  35    Male                  1   
1  0133e584-111e-450a-b451-77a2799ef529  35  Female                  3   
2  0164acd3-34db-4d35-b96c-936daad0ff22  35    Male                  4   
3  01a92062-967a-4900-8dc7-a5ecd3b3f8e2  35  Female                  5   
4  01c878a3-7c3d-457f-99fa-df255960a122  35  Female                  2   

   Seizures  Vision_Problems  Nausea_Vomiting  Memory_Loss  Speech_Difficulty  \
0         0                1                0            1                  0   
1         0                0                0            0                  0   
2         1                0                0            0                  0   
3         0                0                0            0                  0   
4         0                0                0            1                  0   

   Motor_Issues  Family_History  Radiation_

In [10]:
possible_targets = ["Cancer_Risk", "Cancer", "Diagnosis", "Target", "Brain_Cancer", "Outcome"]

target_col = None
for col in possible_targets:
    if col in df.columns:
        target_col = col
        break

if target_col is None:
    raise ValueError("Target column not found. Please check dataset.")

print("Using target column:", target_col)
print(df[target_col].value_counts())

Using target column: Cancer_Risk
Cancer_Risk
1    587
0     30
Name: count, dtype: int64


In [11]:
leakage_cols = [
    "Patient_ID",
    "MRI_Scan_Available",
    "Tumor_Grade",
    "Primary_Diagnosis"
]

leakage_cols = [c for c in leakage_cols if c in df.columns]
df = df.drop(columns=leakage_cols)


In [12]:
# Numeric columns → median
num_cols = df.select_dtypes(include=["int64", "float64"]).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# Categorical columns → mode
cat_cols = df.select_dtypes(include=["object"]).columns
df[cat_cols] = df[cat_cols].fillna(df[cat_cols].mode().iloc[0])


In [13]:
label_encoders = {}

for col in cat_cols:
    if col != target_col:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])
        label_encoders[col] = le


In [14]:
X = df.drop(columns=[target_col])
y = df[target_col]

print("X shape:", X.shape)
print("Target distribution:\n", y.value_counts())


X shape: (617, 17)
Target distribution:
 Cancer_Risk
1    587
0     30
Name: count, dtype: int64


In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [16]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print("scale_pos_weight:", scale_pos_weight)

scale_pos_weight: 0.0511727078891258


In [17]:
model = xgb.XGBClassifier(
    objective="binary:logistic",
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42
)

model.fit(X_train, y_train)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, ...)

In [18]:
probs = model.predict_proba(X_test)[:, 1]

print("Min probability:", probs.min())
print("Max probability:", probs.max())


Min probability: 0.07721561
Max probability: 0.99813724


In [19]:
print("ROC-AUC:", roc_auc_score(y_test, probs))

print("\nConfusion Matrix:\n", confusion_matrix(y_test, (probs >= 0.5).astype(int)))

print("\nClassification Report:\n",
      classification_report(y_test, (probs >= 0.5).astype(int)))


ROC-AUC: 0.53954802259887

Confusion Matrix:
 [[ 0  6]
 [19 99]]

Classification Report:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00         6
           1       0.94      0.84      0.89       118

    accuracy                           0.80       124
   macro avg       0.47      0.42      0.44       124
weighted avg       0.90      0.80      0.84       124



In [20]:
def risk_level(p):
    if p < 0.4:
        return "Low Risk"
    elif p < 0.7:
        return "Medium Risk"
    else:
        return "High Risk"

risk_labels = [risk_level(p) for p in probs]


In [21]:
new_patient = X_train.iloc[0].to_dict()   # safe dummy patient
new_df = pd.DataFrame([new_patient])

prob = model.predict_proba(new_df)[0][1]

print("Cancer Risk Probability:", round(prob, 3))
print("Risk Level:", risk_level(prob))


Cancer Risk Probability: 0.837
Risk Level: High Risk


In [22]:
dummy_low = {
    "Age": 25,
    "Gender": "Female",
    "Headache_Severity": 0,
    "Seizures": 0,
    "Vision_Problems": 0,
    "Nausea_Vomiting": 0,
    "Memory_Loss": 0,
    "Speech_Difficulty": 0,
    "Motor_Issues": 0,
    "Family_History": 0,
    "Radiation_Exposure": 0,
    "Smoking": 0,
    "Alcohol_Consumption": 0,
    "Diet_Type": "Healthy",
    "Physical_Activity_Level": "High",
    "Blood_Pressure": 110,
    "Cholesterol_Level": 160
}


In [28]:
def predict_patient(patient_data):
    # Create a DataFrame from the patient data
    patient_df = pd.DataFrame([patient_data])

    # Apply label encoding to categorical columns
    for col, le in label_encoders.items():
        if col in patient_df.columns:
            # Handle cases where new patient data might have a category not seen before
            # For simplicity, we'll map unseen categories to an existing one or 0
            # A more robust solution might involve handling unknown categories during training
            patient_df[col] = patient_df[col].apply(lambda x: le.transform([x])[0] if x in le.classes_ else -1)
            # If -1 is used, ensure the model can handle it or replace with mode/median
            patient_df[col] = patient_df[col].replace(-1, 0)

    # Ensure the order of columns matches the training data (X_train)
    patient_df = patient_df[X_train.columns]

    # Predict probability
    prob = model.predict_proba(patient_df)[0][1]

    # Get risk level
    risk = risk_level(prob)

    return prob, risk

In [29]:
for name, patient in {
    "LOW": dummy_low
}.items():
    prob, risk = predict_patient(patient)
    print(f"{name} → Probability: {prob:.3f}, Risk: {risk}")


LOW → Probability: 0.468, Risk: Medium Risk


In [33]:
dummy_1= {
    "Age": 45,
    "Gender": "Male",
    "Headache_Severity": 2,
    "Seizures": 0,
    "Vision_Problems": 1,
    "Nausea_Vomiting": 0,
    "Memory_Loss": 0,
    "Speech_Difficulty": 0,
    "Motor_Issues": 0,
    "Family_History": 1,
    "Radiation_Exposure": 0,
    "Smoking": 1,
    "Alcohol_Consumption": 0,
    "Diet_Type": "Balanced",
    "Physical_Activity_Level": "Medium",
    "Blood_Pressure": 135,
    "Cholesterol_Level": 200
}

In [34]:
for name, patient in {
    "Medium": dummy_1
}.items():
    prob, risk = predict_patient(patient)
    print(f"{name} → Probability: {prob:.3f}, Risk: {risk}")

Medium → Probability: 0.237, Risk: Low Risk


In [36]:
dummy_3= {
    "Age": 68,
    "Gender": "Male",
    "Headache_Severity": 4,
    "Seizures": 1,
    "Vision_Problems": 1,
    "Nausea_Vomiting": 1,
    "Memory_Loss": 1,
    "Speech_Difficulty": 1,
    "Motor_Issues": 1,
    "Family_History": 1,
    "Radiation_Exposure": 1,
    "Smoking": 1,
    "Alcohol_Consumption": 1,
    "Diet_Type": "Unhealthy",
    "Physical_Activity_Level": "Low",
    "Blood_Pressure": 160,
    "Cholesterol_Level": 260
}

In [37]:
for name, patient in {
    "Medium": dummy_3
}.items():
    prob, risk = predict_patient(patient)
    print(f"{name} → Probability: {prob:.3f}, Risk: {risk}")

Medium → Probability: 0.673, Risk: Medium Risk


In [38]:
dummy_4 = {
    "Age": 32,
    "Gender": "Female",
    "Headache_Severity": 3,
    "Seizures": 0,
    "Vision_Problems": 1,
    "Nausea_Vomiting": 1,
    "Memory_Loss": 0,
    "Speech_Difficulty": 0,
    "Motor_Issues": 0,
    "Family_History": 0,
    "Radiation_Exposure": 0,
    "Smoking": 0,
    "Alcohol_Consumption": 0,
    "Diet_Type": "Balanced",
    "Physical_Activity_Level": "Medium",
    "Blood_Pressure": 125,
    "Cholesterol_Level": 185
}

In [39]:
for name, patient in {
    "Medium": dummy_4
}.items():
    prob, risk = predict_patient(patient)
    print(f"{name} → Probability: {prob:.3f}, Risk: {risk}")

Medium → Probability: 0.474, Risk: Medium Risk


In [40]:
dummy_6 = {
    "Age": 50,
    "Gender": "Female",
    "Headache_Severity": 2,
    "Seizures": 0,
    "Vision_Problems": 0,
    "Nausea_Vomiting": 0,
    "Memory_Loss": 1,
    "Speech_Difficulty": 0,
    "Motor_Issues": 0,
    "Family_History": 1,
    "Radiation_Exposure": 0,
    "Smoking": 0,
    "Alcohol_Consumption": 0,
    "Diet_Type": "Balanced",
    "Physical_Activity_Level": "Medium",
    "Blood_Pressure": 140,
    "Cholesterol_Level": 210
}

In [41]:
for name, patient in {
    "Medium": dummy_6
}.items():
    prob, risk = predict_patient(patient)
    print(f"{name} → Probability: {prob:.3f}, Risk: {risk}")

Medium → Probability: 0.445, Risk: Medium Risk


In [42]:
dummy_7 = {
    "Age": 55,
    "Gender": "Male",
    "Headache_Severity": 3,
    "Seizures": 1,
    "Vision_Problems": 1,
    "Nausea_Vomiting": 1,
    "Memory_Loss": 1,
    "Speech_Difficulty": 0,
    "Motor_Issues": 1,
    "Family_History": 0,
    "Radiation_Exposure": 1,
    "Smoking": 1,
    "Alcohol_Consumption": 1,
    "Diet_Type": "Unhealthy",
    "Physical_Activity_Level": "Low",
    "Blood_Pressure": 155,
    "Cholesterol_Level": 245
}


In [43]:
for name, patient in {
    "LOW": dummy_7
}.items():
    prob, risk = predict_patient(patient)
    print(f"{name} → Probability: {prob:.3f}, Risk: {risk}")

LOW → Probability: 0.796, Risk: High Risk
